# JAXTPC Pixel Simulation

GPU-accelerated pixel TPC detector simulation.

**Workflow:**
1. Load pixel detector configuration and particle segment data
2. Run simulation (recombination, drift, diffusion, pixel response)
3. Convert output to sparse format
4. Visualize: pixel projections, anode heatmap, waveforms

## Setup and Configuration

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

CONFIG_PATH = "config/cubic_pixel_config.yaml"
DATA_PATH = "out_500.h5"
EVENT_IDX = 7

# Accumulation
USE_BUCKETED = True
MAX_ACTIVE_BUCKETS = 50_000

# Track labeling
INCLUDE_TRACK_HITS = False

# Threshold for sparse output (electrons)
THRESHOLD_ENC = 500

# Performance knobs
TOTAL_PAD = 200_000
RESPONSE_CHUNK_SIZE = 50_000

print("Configuration:")
print(f"  Config: {CONFIG_PATH}")
print(f"  Data: {DATA_PATH}, Event: {EVENT_IDX}")
print(f"  Bucketed: {USE_BUCKETED}, max_buckets: {MAX_ACTIVE_BUCKETS:,}")
print(f"  Track labeling: {'ON' if INCLUDE_TRACK_HITS else 'OFF'}")
print(f"  Threshold: {THRESHOLD_ENC} e-")
print(f"  total_pad: {TOTAL_PAD:,}, chunk: {RESPONSE_CHUNK_SIZE:,}")

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import os
import time
import gc

from tools.simulation import DetectorSimulator
from tools.geometry import generate_detector
from tools.loader import load_event
from tools.config import create_track_hits_config
from tools.output import to_sparse

from tools.pixel_visualization import (
    visualize_pixel_projections,
    visualize_pixel_anode,
    visualize_pixel_waveforms,
    visualize_pixel_buckets,
)

os.makedirs("plots", exist_ok=True)

print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

## Load Data and Create Simulator

In [ ]:
# =============================================================================
# LOAD DATA
# =============================================================================

detector_config = generate_detector(CONFIG_PATH)

jax.clear_caches()
gc.collect()

track_config = create_track_hits_config() if INCLUDE_TRACK_HITS else None

simulator = DetectorSimulator(
    detector_config,
    use_bucketed=USE_BUCKETED,
    max_active_buckets=MAX_ACTIVE_BUCKETS,
    include_track_hits=INCLUDE_TRACK_HITS,
    total_pad=TOTAL_PAD,
    response_chunk_size=RESPONSE_CHUNK_SIZE,
    track_config=track_config,
)

cfg = simulator.config

print(f"\nVolumes: {cfg.n_volumes}")
for v in range(cfg.n_volumes):
    vol = cfg.volumes[v]
    print(f"  Volume {v}: {vol.readout_type}, pixel_shape={vol.pixel_shape}, "
          f"pitch={vol.pixel_pitch_cm*10:.2f}mm")

print(f"\nLoading event {EVENT_IDX} from {DATA_PATH}...")
deposits = load_event(DATA_PATH, cfg, event_idx=EVENT_IDX)

n_volumes = len(deposits.volumes)
n_total = sum(v.n_actual for v in deposits.volumes)
print(f"Loaded {n_total:,} deposits across {n_volumes} volumes")
for v in range(n_volumes):
    print(f"  Volume {v}: {deposits.volumes[v].n_actual:,} deposits")

## Run Simulation

In [ ]:
# =============================================================================
# WARM UP
# =============================================================================

print("Warming up JIT...")
simulator.warm_up()
print("Done.")

In [ ]:
# =============================================================================
# RUN SIMULATION
# =============================================================================

print("Running simulation...")
t0 = time.time()

response_signals, track_hits_raw, deposits = simulator.process_event(
    deposits, key=jax.random.PRNGKey(42))

# Wait for GPU
for val in response_signals.values():
    if isinstance(val, tuple):
        jax.block_until_ready(val[0])
    else:
        jax.block_until_ready(val)

elapsed = time.time() - t0
print(f"Simulation: {elapsed:.2f}s ({n_total / elapsed:,.0f} segments/sec)")

# Show bucket stats
for (vi, pi), sig in response_signals.items():
    if isinstance(sig, tuple) and len(sig) == 6:
        na = int(sig[1])
        print(f"  Volume {vi}: {na:,} active buckets / {MAX_ACTIVE_BUCKETS:,} max")

## Convert to Sparse

In [ ]:
# =============================================================================
# CONVERT TO SPARSE
# =============================================================================

threshold_adc = THRESHOLD_ENC / cfg.electrons_per_adc
t0 = time.time()
sparse_signals = to_sparse(response_signals, cfg, threshold_adc=threshold_adc)
print(f"to_sparse ({THRESHOLD_ENC} e-): {time.time() - t0:.2f}s")

print(f"\nSparse output:")
for (vi, pi), data in sorted(sparse_signals.items()):
    n_vox = len(data['values'])
    total = cfg.volumes[vi].pixel_shape[0] * cfg.volumes[vi].pixel_shape[1] * cfg.num_time_steps
    sparsity = (1 - n_vox / total) * 100
    print(f"  Volume {vi}: {n_vox:,} voxels ({sparsity:.3f}% sparse)")

## Visualize Projections

In [ ]:
# =============================================================================
# THREE ORTHOGONAL PROJECTIONS
# =============================================================================

for v in range(n_volumes):
    fig = visualize_pixel_projections(
        sparse_signals, cfg, vol_idx=v,
        log_norm=True, threshold=threshold_adc, reduce='sum')
    fig.savefig(f"plots/pixel_projections_vol{v}.png", dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()

## Anode Heatmap

In [ ]:
# =============================================================================
# PIXEL ANODE HEATMAP (Y-Z plane, summed over time)
# =============================================================================

for v in range(n_volumes):
    fig = visualize_pixel_anode(
        sparse_signals, cfg, vol_idx=v,
        log_norm=True, threshold=threshold_adc)
    fig.savefig(f"plots/pixel_anode_vol{v}.png", dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()

## Pixel Waveforms

In [ ]:
# =============================================================================
# WAVEFORMS AT SPECIFIC PIXELS
# =============================================================================

# Find the hottest pixels in volume 0
key = (0, 0)
if key in sparse_signals and len(sparse_signals[key]['values']) > 0:
    data = sparse_signals[key]
    py, pz, vals = data['pixel_y'], data['pixel_z'], data['values']
    
    # Aggregate charge per pixel
    pixel_charge = {}
    for i in range(len(vals)):
        k = (int(py[i]), int(pz[i]))
        pixel_charge[k] = pixel_charge.get(k, 0) + abs(float(vals[i]))
    
    top_pixels = sorted(pixel_charge.items(), key=lambda x: x[1], reverse=True)[:6]
    coords = [k for k, _ in top_pixels]
    
    print("Top pixels by charge:")
    for (py_i, pz_i), ch in top_pixels:
        print(f"  ({py_i}, {pz_i}): {ch:,.1f}")
    
    fig = visualize_pixel_waveforms(sparse_signals, cfg, pixel_coords=coords, vol_idx=0)
    fig.savefig("plots/pixel_waveforms.png", dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()
else:
    print("No signal in volume 0.")

## Summary

In [ ]:
# =============================================================================
# SUMMARY
# =============================================================================

print("=" * 60)
print(" PIXEL SIMULATION SUMMARY")
print("=" * 60)

print(f"\nInput:")
print(f"  Data: {DATA_PATH}, Event: {EVENT_IDX}")
print(f"  Total deposits: {n_total:,}")
for v in range(n_volumes):
    print(f"  Volume {v}: {deposits.volumes[v].n_actual:,}")

print(f"\nDetector:")
for v in range(n_volumes):
    vol = cfg.volumes[v]
    print(f"  Volume {v}: {vol.pixel_shape[0]}x{vol.pixel_shape[1]} pixels, "
          f"{vol.pixel_pitch_cm*10:.2f}mm pitch")

print(f"\nSimulation:")
print(f"  Time: {elapsed:.2f}s")
print(f"  Throughput: {n_total / elapsed:,.0f} seg/s")

total_vox = sum(len(d['values']) for d in sparse_signals.values())
total_possible = sum(
    cfg.volumes[v].pixel_shape[0] * cfg.volumes[v].pixel_shape[1] * cfg.num_time_steps
    for v in range(cfg.n_volumes)
)
print(f"\nSparse output ({THRESHOLD_ENC} e-):")
print(f"  Voxels: {total_vox:,}")
print(f"  Sparsity: {(1 - total_vox/total_possible)*100:.3f}%")

print(f"\nSettings:")
print(f"  Bucketed: {USE_BUCKETED}, max_buckets: {MAX_ACTIVE_BUCKETS:,}")
print(f"  Track labeling: {'ON' if INCLUDE_TRACK_HITS else 'OFF'}")